In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("imakash3011/online-shoppers-purchasing-intention-dataset")

print("Path to dataset files:", path)

100%|██████████| 252k/252k [00:00<00:00, 43.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/imakash3011/online-shoppers-purchasing-intention-dataset/versions/1


In [2]:
import pandas as pd
import os

file_path = os.path.join(path, "online_shoppers_intention.csv")
df = pd.read_csv(file_path)

df.head()


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [3]:
df.shape
df.info()
df.describe()
df['Revenue'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

,count
Revenue,
False,10422
True,1908


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Convert target to int
df['Revenue'] = df['Revenue'].astype(int)

# Separate features and target
X = df.drop('Revenue', axis=1)
y = df['Revenue']

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling (important for Logistic + KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [5]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score,
    f1_score, matthews_corrcoef,
    confusion_matrix
)

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    # For AUC (needs probabilities)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred  # fallback

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    cm = confusion_matrix(y_test, y_pred)

    return metrics, cm


# Logistic regression

In [6]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

log_model.fit(X_train_scaled, y_train)

log_metrics, log_cm = evaluate_model(log_model, X_test_scaled, y_test)

print("Logistic Regression Metrics:")
print(log_metrics)
print("Confusion Matrix:\n", log_cm)


Logistic Regression Metrics:
{'Accuracy': 0.8499594484995945, 'AUC': np.float64(0.8962438825858448), 'Precision': 0.5107142857142857, 'Recall': 0.7486910994764397, 'F1 Score': 0.6072186836518046, 'MCC': np.float64(0.5330405682140082)}
Confusion Matrix:
 [[1810  274]
 [  96  286]]


# Decision Tree Classifier

In [7]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

dt_model.fit(X_train, y_train)

dt_metrics, dt_cm = evaluate_model(dt_model, X_test, y_test)

print("Decision Tree Metrics:")
print(dt_metrics)
print("Confusion Matrix:\n", dt_cm)


Decision Tree Metrics:
{'Accuracy': 0.8544201135442011, 'AUC': np.float64(0.7171757393654974), 'Precision': 0.5308310991957105, 'Recall': 0.518324607329843, 'F1 Score': 0.5245033112582781, 'MCC': np.float64(0.4386143266117872)}
Confusion Matrix:
 [[1909  175]
 [ 184  198]]


# K-Nearest Neighbors

In [8]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

knn_metrics, knn_cm = evaluate_model(knn_model, X_test_scaled, y_test)

print("KNN Metrics:")
print(knn_metrics)
print("Confusion Matrix:\n", knn_cm)


KNN Metrics:
{'Accuracy': 0.8686131386861314, 'AUC': np.float64(0.7724592004903981), 'Precision': 0.6367924528301887, 'Recall': 0.35340314136125656, 'F1 Score': 0.45454545454545453, 'MCC': np.float64(0.4084581500453135)}
Confusion Matrix:
 [[2007   77]
 [ 247  135]]


# Naive Bayes (Gaussian)

In [9]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

nb_model.fit(X_train_scaled, y_train)

nb_metrics, nb_cm = evaluate_model(nb_model, X_test_scaled, y_test)

print("Naive Bayes Metrics:")
print(nb_metrics)
print("Confusion Matrix:\n", nb_cm)


Naive Bayes Metrics:
{'Accuracy': 0.6763990267639902, 'AUC': np.float64(0.7974262895559285), 'Precision': 0.29766536964980544, 'Recall': 0.8010471204188482, 'F1 Score': 0.4340425531914894, 'MCC': np.float64(0.33360461043770534)}
Confusion Matrix:
 [[1362  722]
 [  76  306]]


# Random Forest (Ensemble)

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_metrics, rf_cm = evaluate_model(rf_model, X_test, y_test)

print("Random Forest Metrics:")
print(rf_metrics)
print("Confusion Matrix:\n", rf_cm)


Random Forest Metrics:
{'Accuracy': 0.8965936739659367, 'AUC': np.float64(0.9187954095527128), 'Precision': 0.7432950191570882, 'Recall': 0.5078534031413613, 'F1 Score': 0.6034214618973561, 'MCC': np.float64(0.559490232887937)}
Confusion Matrix:
 [[2017   67]
 [ 188  194]]


# XGBoost (Ensemble Boosting)

In [11]:
pip install xgboost


In [12]:
from xgboost import XGBClassifier

scale_pos_weight = 10422 / 1908  # class imbalance ratio

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_metrics, xgb_cm = evaluate_model(xgb_model, X_test, y_test)

print("XGBoost Metrics:")
print(xgb_metrics)
print("Confusion Matrix:\n", xgb_cm)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [12:10:51] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Metrics:
{'Accuracy': 0.8811841038118411, 'AUC': np.float64(0.9224181246294381), 'Precision': 0.5913757700205339, 'Recall': 0.7539267015706806, 'F1 Score': 0.6628308400460299, 'MCC': np.float64(0.5984220104491095)}
Confusion Matrix:
 [[1885  199]
 [  94  288]]


In [13]:
import pandas as pd

results = pd.DataFrame({
    "Logistic Regression": log_metrics,
    "Decision Tree": dt_metrics,
    "KNN": knn_metrics,
    "Naive Bayes": nb_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics
}).T

results


,Accuracy,AUC,Precision,Recall,F1 Score,MCC
Logistic Regression,0.849959,0.896244,0.510714,0.748691,0.607219,0.533041
Decision Tree,0.854420,0.717176,0.530831,0.518325,0.524503,0.438614
KNN,0.868613,0.772459,0.636792,0.353403,0.454545,0.408458
Naive Bayes,0.676399,0.797426,0.297665,0.801047,0.434043,0.333605
Random Forest,0.896594,0.918795,0.743295,0.507853,0.603421,0.559490
XGBoost,0.881184,0.922418,0.591376,0.753927,0.662831,0.598422


In [14]:
import joblib
import os

# Create model directory
os.makedirs("model", exist_ok=True)

# Save models
joblib.dump(log_model, "model/logistic_regression.pkl")
joblib.dump(dt_model, "model/decision_tree.pkl")
joblib.dump(knn_model, "model/knn.pkl")
joblib.dump(nb_model, "model/naive_bayes.pkl")
joblib.dump(rf_model, "model/random_forest.pkl")
joblib.dump(xgb_model, "model/xgboost.pkl")

# Save scaler
joblib.dump(scaler, "model/scaler.pkl")

# Save feature columns
joblib.dump(X.columns.tolist(), "model/feature_columns.pkl")

print("All models and preprocessing files saved successfully.")


All models and preprocessing files saved successfully.


In [16]:
import pkg_resources
import re

# Get a list of all installed packages and their versions
installed_packages = {pkg.key: pkg.version for pkg in pkg_resources.working_set}

def extract_libraries_from_code():
    used_libraries = set()
    code_cells = [
        "import kagglehub",
        "import pandas as pd",
        "import os",
        "from sklearn.model_selection import train_test_split",
        "from sklearn.preprocessing import StandardScaler",
        "from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix)",
        "from sklearn.linear_model import LogisticRegression",
        "from sklearn.tree import DecisionTreeClassifier",
        "from sklearn.neighbors import KNeighborsClassifier",
        "from sklearn.naive_bayes import GaussianNB",
        "from sklearn.ensemble import RandomForestClassifier",
        "from xgboost import XGBClassifier",
        "import joblib"
    ]

    for cell_content in code_cells:
        # Regex to find 'import <library>' or 'from <library> import ...'
        matches = re.findall(r'^(?:import|from)\s+([a-zA-Z0-9_.]+)', cell_content, re.MULTILINE)
        for match in matches:
            # Take the top-level package name
            top_level_package = match.split('.')[0]
            used_libraries.add(top_level_package.lower())
    return used_libraries

used_libraries = extract_libraries_from_code()

# Map common aliases or specific library names if needed
alias_map = {
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'xgboost': 'xgboost',
    'joblib': 'joblib',
    'kagglehub': 'kagglehub'
}

# Filter installed packages to only include used libraries and get their versions
requirements = []
for lib in used_libraries:
    # Check if the library or its alias is in the installed packages
    package_name = alias_map.get(lib, lib) # Use alias if available
    if package_name in installed_packages:
        requirements.append(f"{package_name}=={installed_packages[package_name]}")
    else:
        # If not found directly, try to be flexible with common names
        if lib == 'numpy' and 'numpy' in installed_packages:
            requirements.append(f"numpy=={installed_packages['numpy']}")
        elif lib == 'scipy' and 'scipy' in installed_packages:
            requirements.append(f"scipy=={installed_packages['scipy']}")
        elif lib == 'setuptools' and 'setuptools' in installed_packages:
            requirements.append(f"setuptools=={installed_packages['setuptools']}")

# Add specific packages that might not be directly imported but are dependencies
# For example, numpy and scipy are often dependencies of scikit-learn
if 'scikit-learn' in installed_packages and 'numpy' not in installed_packages:
    if 'numpy' in installed_packages: # Double-check to avoid duplicates
        requirements.append(f"numpy=={installed_packages['numpy']}")
if 'scikit-learn' in installed_packages and 'scipy' not in installed_packages:
    if 'scipy' in installed_packages: # Double-check to avoid duplicates
        requirements.append(f"scipy=={installed_packages['scipy']}")

# Sort requirements alphabetically
requirements.sort()

# Write to requirements.txt
with open("requirements.txt", "w") as f:
    for req in requirements:
        f.write(req + "\n")

print("requirements.txt created successfully with the following content:")
for req in requirements:
    print(req)

requirements.txt created successfully with the following content:
joblib==1.5.3
kagglehub==0.3.13
pandas==2.2.2
scikit-learn==1.6.1
xgboost==3.1.3


/tmp/ipython-input-2698162377.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [15]:
# Combine X_test and y_test for evaluation
test_df = X_test.copy()
test_df['Revenue'] = y_test

# Save test file
test_df.to_csv("test.csv", index=False)

print("test.csv created successfully.")


test.csv created successfully.
